In [1]:
!pip install gradio transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.5 MB/s eta 0:00:00


In [2]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from google.colab import drive

# ربط Google Drive
drive.mount('/content/drive')

# تحميل النموذج والـ tokenizer
model_path = "/content/drive/MyDrive/model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Mounted at /content/drive


In [3]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from io import BytesIO
from PIL import Image

# اكواد صح وليست ضرورية

In [4]:

def classify_review(text):
    # تجهيز النص
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)

    # تمرير للنموذج
    with torch.no_grad():
        outputs = model(**inputs)

    # حساب الاحتمالات
    logits = outputs.logits
    probs = F.softmax(logits, dim=1).squeeze().numpy()

    # استخراج التوقع الأعلى
    predicted_class = np.argmax(probs)
    confidence = probs[predicted_class]

    # تسميات التصنيفات
    label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
    labels = [label_map[i] for i in range(len(probs))]

    # النص الناتج
    prediction_text = f"{labels[predicted_class]} ({confidence * 100:.2f}%)"

    # رسم المخطط
    fig, ax = plt.subplots()
    bars = ax.bar(labels, probs * 100, color=['red', 'gray', 'green'])
    ax.set_ylim([0, 100])
    ax.set_ylabel("Confidence (%)")
    ax.set_title("Sentiment Prediction Probabilities")
    for bar in bars:
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.1f}%", ha='center', va='bottom')

    # تحويل الرسم إلى صورة
    buf = BytesIO()
    plt.savefig(buf, format='png')
    buf.seek(0)
    img = Image.open(buf)
    plt.close()

    return prediction_text, img

In [12]:
import gradio as gr

description = description = """
Created by :  **Wjdan Al-Harthi**\n
\n
A student at Ironhack, As part of an NLP project analyzing customer reviews.\n
\n
This interface is part of an NLP-based project that automates the analysis of customer reviews.\n
It uses a fine-tuned transformer model to classify reviews into Positive, Neutral, or Negative sentiments with high accuracy.\n
\n
Model performance:\n
• Accuracy: 96.7%\n
• Precision: 96.6%\n
• Recall: 96.7%\n
• F1-score: 96.6%\n
\n
The system helps businesses extract valuable insights and improve product decision-making based on real customer feedback.
"""


iface = gr.Interface(
    fn=classify_review,
    inputs=gr.Textbox(label="Enter your review"),
    outputs=[
        gr.Label(label="Sentiment & Confidence"),
        gr.Image(type="pil", label="Prediction Chart")
    ],
    title="Review Sentiment Classifier",
    description=description
)

iface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://13c8a006bed13e5c5a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# نبدأ من هينا

In [7]:
import joblib
kmeans_model = joblib.load('/content/drive/MyDrive/kmeans_model.pkl')

from sentence_transformers import SentenceTransformer
model = SentenceTransformer.load('/content/drive/MyDrive/sentence_transformer_model')

model_path = "/content/drive/MyDrive/model"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [8]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# المسار إلى المجلد الذي يحتوي على النموذج
model_save_path = '/content/drive/MyDrive/Last_model_folder/'

# تحميل tokenizer والنموذج من المسار المحلي
tokenizer = AutoTokenizer.from_pretrained(model_save_path, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_save_path, local_files_only=True)

# الآن النموذج والـtokenizer جاهزان للاستخدام


/usr/local/lib/python3.11/dist-packages/transformers/models/bart/configuration_bart.py:176: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(
